# 02 Carry Research — Macro Metals System

> **Strategy:** Carry / Term-Structure (Memory File §3.2)
> **Scope:** In-sample development (2015–2022)
> **Sub-modules:** FX Cross-Sectional Carry · Metals Calendar Spreads · SOFR Curve
> **Integration:** Carry-with-trend filter from canonical TSMOM sleeve
>
> **How carry differs from TSMOM:**
> - TSMOM is *time-series*: sign(own 12M return) per instrument.
> - Carry is *cross-sectional* (rank instruments by implied yield) and
>   *term-structure* (calendar spreads, curve slope).
> - Carry and momentum are complementary premia with low correlation,
>   making them natural portfolio partners.
>
> Self-contained BQuant notebook — executable top-to-bottom.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from pathlib import Path
from datetime import datetime
from typing import Dict, Tuple, Optional

# Bloomberg BQL
import bql
bq = bql.Service()

print(f"Session started : {datetime.now():%Y-%m-%d %H:%M}")
print(f"BQL service     : {type(bq).__name__}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

## Config & Parameters

Load carry parameters from `parameters.yaml` (`strategies.carry` section).
Three sub-modules with distinct rebalance frequencies:

| Sub-module | Rebalance | Key params |
|------------|-----------|------------|
| FX Carry | Weekly | 3M horizon, rank top/bot 30%, trend filter 63d, VIX crisis 30 |
| Metals Spreads | Weekly | 120d z-score, entry ±1.5σ, exit ±0.5σ |
| SOFR Curve | Weekly | 60d z-score, 10bp slope threshold |

In [ ]:
CONFIG_DIR = Path("config")

with open(CONFIG_DIR / "parameters.yaml") as f:
    params = yaml.safe_load(f)
with open(CONFIG_DIR / "tickers.yaml") as f:
    tickers = yaml.safe_load(f)

gcfg       = params["global"]
carry_cfg  = params["strategies"]["carry"]
targets    = params["performance_targets"]

fx_cfg   = carry_cfg["fx"]
met_cfg  = carry_cfg["metals"]["calendar_spreads"]
sofr_cfg = carry_cfg["sofr_curve"]

IS_START = gcfg["in_sample_start"]
IS_END   = gcfg["in_sample_end"]

# Common carry parameters
TARGET_VOL       = gcfg.get("target_portfolio_vol_annual", 0.10)
VOL_LAMBDA       = gcfg.get("vol_decay_lambda", 0.94)
LEV_CAP          = gcfg.get("vol_cap_multiplier", 2.0)
TC_BP            = 2.0
TREND_FILTER_MODE = "soft"   # "hard" = zero weight, "soft" = 0.5x weight

# All submodules rebalance weekly (frozen within week)
REBAL_FREQ       = "W-FRI"

print("Carry parameters:")
print(f"\n  FX Carry:")
print(f"    Pairs            : {fx_cfg['pairs']}")
print(f"    Horizon          : {fx_cfg['carry_horizon_days']}d")
print(f"    Rank top/bot     : {fx_cfg['ranking_top_pct']:.0%} / {fx_cfg['ranking_bottom_pct']:.0%}")
print(f"    Min carry        : {fx_cfg['min_carry_bp']} bp")
print(f"    Trend filter     : {fx_cfg['trend_filter_lookback_days']}d")
print(f"    Crisis VIX       : {fx_cfg['crisis_vix_threshold']}")

print(f"\n  Metals Calendar Spreads:")
print(f"    Instruments      : {met_cfg['instruments']}")
print(f"    Z-score window   : {met_cfg['zscore_lookback_days']}d")
print(f"    Entry/exit z     : ±{met_cfg['entry_zscore']} / ±{met_cfg['exit_zscore']}")

print(f"\n  SOFR Curve:")
print(f"    Instruments      : {sofr_cfg['instruments']}")
print(f"    Z-score window   : {sofr_cfg['zscore_lookback_days']}d")
print(f"    Slope thresh.    : {sofr_cfg['steepness_threshold_bp']} bp")

print(f"\n  Common:")
print(f"    Vol target       : {TARGET_VOL:.0%}")
print(f"    EWMA lambda      : {VOL_LAMBDA}")
print(f"    TC per side      : {TC_BP:.1f} bp")
print(f"    Trend filter     : {TREND_FILTER_MODE}")
print(f"    Rebalance        : weekly ({REBAL_FREQ})")
print(f"    IS period        : {IS_START} to {IS_END}")

## Data Loader (BQL)

`BQuantDataLoader` with methods for:
- Single ticker history
- Multi-ticker panel (curve points, spread components)
- FX spot + forward pair

Uses corrected BQL syntax: `df.set_index('DATE')`, dedup, `pd.to_datetime`.

In [ ]:
class BQuantDataLoader:
    """Fetch historical prices via Bloomberg BQL."""

    def __init__(self, ticker_map: dict) -> None:
        self._tickers = ticker_map
        self._bq = bql.Service()

    def _resolve(self, logical_name: str) -> Optional[str]:
        """Logical name -> Bloomberg ticker string, or None."""
        for group in self._tickers.values():
            if isinstance(group, dict) and logical_name in group:
                return group[logical_name]
        return None

    def get_history(self, logical_name: str, start: str, end: str,
                    field: str = "PX_LAST") -> pd.Series:
        """Fetch daily price series for one instrument."""
        bbg = self._resolve(logical_name)
        if bbg is None:
            print(f"  ! {logical_name}: not in tickers.yaml")
            return pd.Series(dtype=float, name=logical_name)
        request = bql.Request(
            bbg,
            {field: self._bq.data.px_last(
                dates=self._bq.func.range(start, end)
            )},
        )
        try:
            response = self._bq.execute(request)
            df = response[0].df()
            if df.empty:
                print(f"  ! {logical_name}: empty response")
                return pd.Series(dtype=float, name=logical_name)
            df_fixed = df.set_index('DATE')
            series = df_fixed[field]
            series.index = pd.to_datetime(series.index, errors='coerce')
            series = series.dropna()
            series = series[~series.index.duplicated(keep='last')]
            series = series.sort_index().astype(float)
            series.name = logical_name
            series.index.name = "date"
            return series
        except Exception as exc:
            print(f"  ! {logical_name} ({bbg}): {exc}")
            return pd.Series(dtype=float, name=logical_name)

    def get_multi_history(self, logical_names: list, start: str, end: str,
                          field: str = "PX_LAST") -> pd.DataFrame:
        """Fetch panel of prices for multiple tickers."""
        panel = {}
        for name in logical_names:
            s = self.get_history(name, start, end, field)
            if len(s) > 0:
                panel[name] = s
        df = pd.DataFrame(panel).sort_index()
        df = df[df.index.notna()].ffill()
        # Drop all-NaN columns
        df = df.dropna(axis=1, how="all")
        return df

    def get_fx_spot_and_fwd(self, spot_name: str, fwd_name: str,
                            start: str, end: str) -> pd.DataFrame:
        """Fetch FX spot + 3M forward outright."""
        spot = self.get_history(spot_name, start, end)
        fwd  = self.get_history(fwd_name, start, end)
        df = pd.DataFrame({"spot": spot, "forward": fwd}).dropna()
        df["fwd_points"] = df["forward"] - df["spot"]
        return df

    def get_curve_points(self, logical_names: list, start: str,
                         end: str) -> pd.DataFrame:
        """Alias for get_multi_history (curve context)."""
        return self.get_multi_history(logical_names, start, end)


loader = BQuantDataLoader(tickers)
print("BQuantDataLoader ready.")

## Carry Universe & Breadth Report

Three sub-universes:
1. **FX Carry:** G10 pairs with 3M forwards available
2. **Metals Spreads:** GC and SI front-3 calendar spreads
3. **SOFR Curve:** Front 4 quarterly SOFR futures

In [ ]:
# ── FX Carry: spot + forward pairs ────────────────────────────────
FX_PAIRS = {
    "eurusd": ("eurusd_spot", "eurusd_3m_fwd"),
    "usdjpy": ("usdjpy_spot", "usdjpy_3m_fwd"),
    "gbpusd": ("gbpusd_spot", "gbpusd_3m_fwd"),
    "audusd": ("audusd_spot", "audusd_3m_fwd"),
    "usdcnh": ("usdcnh_spot", "usdcnh_3m_fwd"),
}

# Map config pair names to our keys
FX_PAIR_MAP = {
    "eurusd_spot": "eurusd", "usdjpy_spot": "usdjpy",
    "gbpusd_spot": "gbpusd", "audusd_spot": "audusd",
    "usdchf_spot": None,     "usdcnh_spot": "usdcnh",
}

print("=== FX Carry Data ===")
fx_data = {}
for pair_cfg in fx_cfg["pairs"]:
    key = FX_PAIR_MAP.get(pair_cfg)
    if key is None:
        print(f"  {pair_cfg:18s} SKIP (no forward ticker)")
        continue
    spot_name, fwd_name = FX_PAIRS[key]
    print(f"  {key:18s}", end=" ")
    try:
        df = loader.get_fx_spot_and_fwd(spot_name, fwd_name, IS_START, IS_END)
        fx_data[key] = df
        print(f"OK  {len(df):>5d} obs  [{df.index[0]:%Y-%m-%d} -> {df.index[-1]:%Y-%m-%d}]")
    except Exception as e:
        print(f"FAIL: {e}")

# Also fetch spot-only for trend filter
fx_spot_df = pd.DataFrame({k: v["spot"] for k, v in fx_data.items()}).ffill()

# ── Metals futures (front/second/third for GC and SI) ─────────────
print("\n=== Metals Calendar Spread Data ===")
metals_tickers = met_cfg["instruments"]
metals_prices = loader.get_curve_points(metals_tickers, IS_START, IS_END)
print(f"  Panel: {metals_prices.shape[0]} days x {metals_prices.shape[1]} contracts")

# ── SOFR futures ──────────────────────────────────────────────────
print("\n=== SOFR Curve Data ===")
sofr_tickers = sofr_cfg["instruments"]
sofr_prices = loader.get_curve_points(sofr_tickers, IS_START, IS_END)
print(f"  Panel: {sofr_prices.shape[0]} days x {sofr_prices.shape[1]} contracts")

# ── VIX for crisis filter ────────────────────────────────────────
print("\n=== Risk Indicators ===")
vix = loader.get_history("vix", IS_START, IS_END)
print(f"  VIX: {len(vix)} obs")

# ── Breadth report ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  BREADTH REPORT")
print("=" * 65)
print(f"  FX Carry       : {len(fx_data)}/{len(FX_PAIRS)} pairs with spot+fwd")
print(f"  Metals Spreads : {metals_prices.shape[1]}/{len(metals_tickers)} contracts")
print(f"  SOFR Curve     : {sofr_prices.shape[1]}/{len(sofr_tickers)} contracts")
print(f"  VIX            : {'OK' if len(vix) > 0 else 'MISSING'}")

## TSMOM Trend Filter Integration

Import the canonical TSMOM monthly sign signal to use as a **carry filter**:
- If TSMOM file exists (`outputs/tsmom_signals_monthly_*.csv`), load it.
- Otherwise, compute minimal 12M sign filter for carry underlyings.
- Trend filter mode: **soft** (scale carry weight by 0.5 when misaligned) or
  **hard** (zero out misaligned carry positions).

This ensures carry and momentum do not double-count the same directional bets.

In [ ]:
def load_or_compute_trend_filter(
    prices_df: pd.DataFrame,
    output_dir: Path = Path("../outputs"),
    lookback: int = 252,
) -> pd.DataFrame:
    """Load TSMOM signals or compute minimal 12M sign filter.

    Returns DataFrame of daily signals in {-1, 0, +1}.
    """
    # Try to load from TSMOM exports
    import glob
    pattern = str(output_dir / "tsmom_signals_monthly_*.csv")
    files = sorted(glob.glob(pattern))
    if files:
        latest = files[-1]
        print(f"  Loaded TSMOM signals from: {Path(latest).name}")
        sig = pd.read_csv(latest, index_col=0, parse_dates=True)
        # Reindex to match our price panel
        sig = sig.reindex(prices_df.index).ffill().fillna(0.0)
        return sig

    # Fallback: compute minimal 12M sign, monthly frozen
    print("  TSMOM export not found — computing minimal trend filter")
    r12 = prices_df.pct_change(lookback)
    sig_daily = np.sign(r12).shift(1)
    sig_monthly = sig_daily.resample("M").last().shift(1)
    sig = sig_monthly.reindex(prices_df.index).ffill().fillna(0.0)
    return sig


# Build a combined price panel for trend filter computation
# We need trend on: FX spot pairs, metals fronts (GC, SI), SOFR front
trend_prices = {}
for pair, df in fx_data.items():
    trend_prices[pair] = df["spot"]
# Add metals front contracts
for inst in ["gc_fut_front", "si_fut_front"]:
    if inst in metals_prices.columns:
        trend_prices[inst] = metals_prices[inst]
# Add SOFR front
if "sofr_fut_front" in sofr_prices.columns:
    trend_prices["sofr_fut_front"] = sofr_prices["sofr_fut_front"]

trend_prices_df = pd.DataFrame(trend_prices).sort_index().ffill()
trend_filter_daily = load_or_compute_trend_filter(trend_prices_df)

print(f"\n  Trend filter shape: {trend_filter_daily.shape}")
print(f"  Columns: {list(trend_filter_daily.columns)}")

## Carry Strategy

Three submodules with a common interface:
- **A) FX Carry:** Cross-sectional ranking of implied carry, trend-filtered, weekly frozen
- **B) Metals Spreads:** Z-score entry/exit with hysteresis, trend-scaled
- **C) SOFR Curve:** Slope z-scores, butterfly, weekly frozen

All submodules:
- Freeze weights at weekly rebalance dates (no intra-week resizing)
- Vol-scale using EWMA(λ=0.94) sampled at rebalance and frozen
- Apply trend filter (soft: 0.5× when misaligned)

In [ ]:
def ewma_vol_ann(returns: pd.Series, lam: float = 0.94) -> pd.Series:
    """EWMA annualised volatility (RiskMetrics)."""
    alpha = 1 - lam
    ewma_var = returns.pow(2).ewm(alpha=alpha, adjust=False).mean()
    return np.sqrt(ewma_var) * np.sqrt(252)


def freeze_weekly(series: pd.DataFrame, freq: str = "W-FRI") -> pd.DataFrame:
    """Sample at weekly frequency and forward-fill (freeze within week)."""
    weekly = series.resample(freq).last()
    return weekly.reindex(series.index).ffill()


def apply_trend_filter(
    weights: pd.DataFrame,
    trend: pd.DataFrame,
    col_map: dict,
    mode: str = "soft",
) -> pd.DataFrame:
    """Apply TSMOM trend filter to carry weights.

    col_map: maps weight column -> trend column name.
    mode: "soft" (0.5x when misaligned) or "hard" (0x).
    """
    filtered = weights.copy()
    scale = 0.0 if mode == "hard" else 0.5
    for w_col, t_col in col_map.items():
        if w_col not in filtered.columns or t_col not in trend.columns:
            continue
        t = trend[t_col].reindex(filtered.index).ffill().fillna(0)
        # Misaligned: carry weight sign != trend sign (and both non-zero)
        misaligned = (np.sign(filtered[w_col]) != 0) & (np.sign(t) != 0) & \
                     (np.sign(filtered[w_col]) != np.sign(t))
        filtered.loc[misaligned, w_col] *= scale
    return filtered


class CarryStrategy:
    """Carry / term-structure strategy (§3.2) with 3 sub-modules."""

    def __init__(self, params: dict, tgt_vol: float, vol_lam: float,
                 lev_cap: float, rebal_freq: str) -> None:
        self.fx_cfg   = params["strategies"]["carry"]["fx"]
        self.met_cfg  = params["strategies"]["carry"]["metals"]["calendar_spreads"]
        self.sofr_cfg = params["strategies"]["carry"]["sofr_curve"]
        self.tgt_vol  = tgt_vol
        self.vol_lam  = vol_lam
        self.lev_cap  = lev_cap
        self.rebal    = rebal_freq

    # ══════════════════════════════════════════════════════════════
    # A) FX CARRY (cross-sectional)
    # ══════════════════════════════════════════════════════════════
    def fx_carry_signals(
        self,
        fx_data: dict,
        vix: pd.Series,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Generate FX carry signals and vol-scaled weights.

        Returns: (raw_signals, weights, carry_matrix)
        """
        horizon = self.fx_cfg["carry_horizon_days"]
        top_pct = self.fx_cfg["ranking_top_pct"]
        bot_pct = self.fx_cfg["ranking_bottom_pct"]
        min_bp  = self.fx_cfg["min_carry_bp"]
        vix_thr = self.fx_cfg["crisis_vix_threshold"]
        use_crisis = self.fx_cfg.get("crisis_filter", True)

        # 1. Annualised implied carry
        carry_df = pd.DataFrame()
        spot_df  = pd.DataFrame()
        for pair, df in fx_data.items():
            implied = (-df["fwd_points"] / df["spot"]) * (365 / horizon)
            carry_df[pair] = implied
            spot_df[pair]  = df["spot"]

        idx = carry_df.dropna(how="all").index
        carry_df = carry_df.reindex(idx)
        spot_df  = spot_df.reindex(idx)
        vix_a    = vix.reindex(idx).ffill()

        # 2. Cross-sectional ranking
        ranks = carry_df.rank(axis=1, pct=True)
        signals = pd.DataFrame(0.0, index=idx, columns=carry_df.columns)

        for col in carry_df.columns:
            sig = pd.Series(0.0, index=idx)
            sig[ranks[col] >= (1 - top_pct)] =  1.0
            sig[ranks[col] <= bot_pct]        = -1.0
            # Min carry filter
            sig[carry_df[col].abs() * 10_000 < min_bp] = 0.0
            signals[col] = sig

        # 3. Crisis filter
        if use_crisis:
            crisis = vix_a > vix_thr
            signals.loc[crisis] = 0.0

        # 4. Freeze weekly
        signals = freeze_weekly(signals, self.rebal)

        # 5. Vol-scale: EWMA on spot returns, sampled weekly
        spot_ret = spot_df.pct_change().fillna(0.0)
        vol_ann = pd.DataFrame({
            col: ewma_vol_ann(spot_ret[col], self.vol_lam) for col in spot_df.columns
        })
        vol_weekly = freeze_weekly(vol_ann, self.rebal)
        vol_weekly = vol_weekly.replace(0, np.nan)

        weights = signals * (self.tgt_vol / vol_weekly)
        cap = self.lev_cap * (self.tgt_vol / vol_weekly)
        weights = weights.clip(-cap, cap).fillna(0.0)

        return signals, weights, carry_df

    # ══════════════════════════════════════════════════════════════
    # B) METALS CALENDAR SPREADS
    # ══════════════════════════════════════════════════════════════
    def metals_spread_signals(
        self,
        prices: pd.DataFrame,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Generate metals calendar spread z-score signals.

        Returns: (signals, weights, zscores)
        """
        zs_lb   = self.met_cfg["zscore_lookback_days"]
        entry_z = self.met_cfg["entry_zscore"]
        exit_z  = self.met_cfg["exit_zscore"]
        mom_lb  = self.met_cfg["carry_momentum_lookback_days"]

        spread_pairs = {
            "GC_1v2": ("gc_fut_front", "gc_fut_second"),
            "GC_2v3": ("gc_fut_second", "gc_fut_third"),
            "SI_1v2": ("si_fut_front", "si_fut_second"),
            "SI_2v3": ("si_fut_second", "si_fut_third"),
        }

        signals  = pd.DataFrame(index=prices.index)
        zscores  = pd.DataFrame(index=prices.index)
        spreads  = pd.DataFrame(index=prices.index)

        for label, (front, back) in spread_pairs.items():
            if front not in prices.columns or back not in prices.columns:
                continue

            spread = prices[front] - prices[back]
            spreads[label] = spread

            # Rolling z-score
            mu  = spread.rolling(zs_lb, min_periods=zs_lb // 2).mean()
            sig = spread.rolling(zs_lb, min_periods=zs_lb // 2).std().replace(0, np.nan)
            z   = (spread - mu) / sig
            zscores[label] = z

            # Hysteresis signal
            signal = pd.Series(0.0, index=prices.index)
            pos = 0.0
            for i in range(len(z)):
                if pd.isna(z.iloc[i]):
                    signal.iloc[i] = 0.0
                    continue
                zv = z.iloc[i]
                if pos == 0.0:
                    if zv > entry_z:
                        pos = -1.0
                    elif zv < -entry_z:
                        pos = 1.0
                elif pos > 0 and zv > -exit_z:
                    pos = 0.0
                elif pos < 0 and zv < exit_z:
                    pos = 0.0
                signal.iloc[i] = pos

            signals[label] = signal

        # Freeze weekly
        signals = freeze_weekly(signals, self.rebal)

        # Vol-scale on spread returns (dollar-vol proxy)
        spread_ret = spreads.pct_change().fillna(0.0)
        vol_ann = pd.DataFrame({
            col: ewma_vol_ann(spread_ret[col], self.vol_lam)
            for col in spreads.columns if col in signals.columns
        })
        vol_weekly = freeze_weekly(vol_ann, self.rebal).replace(0, np.nan)

        weights = signals * (self.tgt_vol / vol_weekly)
        cap = self.lev_cap * (self.tgt_vol / vol_weekly)
        weights = weights.clip(-cap, cap).fillna(0.0)

        return signals, weights, zscores

    # ══════════════════════════════════════════════════════════════
    # C) SOFR CURVE
    # ══════════════════════════════════════════════════════════════
    def sofr_curve_signals(
        self,
        prices: pd.DataFrame,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Generate SOFR curve slope/butterfly signals.

        Returns: (signals, weights, slopes)
        """
        zs_lb     = self.sofr_cfg["zscore_lookback_days"]
        stability = self.sofr_cfg["policy_stability_window_days"]
        contracts = [c for c in self.sofr_cfg["instruments"] if c in prices.columns]

        if len(contracts) < 2:
            empty = pd.DataFrame(index=prices.index)
            return empty, empty, empty

        slopes  = pd.DataFrame(index=prices.index)
        signals = pd.DataFrame(index=prices.index)

        # Adjacent slopes
        for i in range(len(contracts) - 1):
            f, b = contracts[i], contracts[i + 1]
            label = f"SOFR_{i+1}v{i+2}"
            # In rate space: slope = (100-f) - (100-b) = b - f
            slope = prices[b] - prices[f]
            slopes[label] = slope

            mu  = slope.rolling(zs_lb, min_periods=zs_lb // 2).mean()
            sig = slope.rolling(zs_lb, min_periods=zs_lb // 2).std().replace(0, np.nan)
            z = (slope - mu) / sig
            signal = np.tanh(z)  # tanh allowed for rates carry

            # Policy stability filter
            slope_vol = slope.rolling(stability).std()
            sv_mu  = slope_vol.rolling(zs_lb).mean()
            sv_sig = slope_vol.rolling(zs_lb).std().replace(0, np.nan)
            sv_z   = (slope_vol - sv_mu) / sv_sig
            signal[sv_z.abs() > 2.0] *= 0.5

            signals[label] = signal

        # Butterfly (if 3+ contracts)
        if len(contracts) >= 3:
            fly = prices[contracts[0]] - 2 * prices[contracts[1]] + prices[contracts[2]]
            slopes["SOFR_fly"] = fly
            mu  = fly.rolling(zs_lb, min_periods=zs_lb // 2).mean()
            sig = fly.rolling(zs_lb, min_periods=zs_lb // 2).std().replace(0, np.nan)
            signals["SOFR_fly"] = np.tanh((fly - mu) / sig)

        # Freeze weekly
        signals = freeze_weekly(signals, self.rebal)

        # Vol-scale
        slope_ret = slopes.pct_change().fillna(0.0)
        vol_ann = pd.DataFrame({
            col: ewma_vol_ann(slope_ret[col], self.vol_lam)
            for col in slopes.columns if col in signals.columns
        })
        vol_weekly = freeze_weekly(vol_ann, self.rebal).replace(0, np.nan)

        weights = signals * (self.tgt_vol / vol_weekly)
        cap = self.lev_cap * (self.tgt_vol / vol_weekly)
        weights = weights.clip(-cap, cap).fillna(0.0)

        return signals, weights, slopes

    # ══════════════════════════════════════════════════════════════
    # run_all
    # ══════════════════════════════════════════════════════════════
    def run_all(self, fx_data, metals_prices, sofr_prices, vix):
        """Run all 3 submodules, return structured results."""
        fx_sig, fx_w, fx_carry   = self.fx_carry_signals(fx_data, vix)
        mt_sig, mt_w, mt_z       = self.metals_spread_signals(metals_prices)
        sf_sig, sf_w, sf_slopes  = self.sofr_curve_signals(sofr_prices)
        return {
            "fx":    {"signals": fx_sig, "weights": fx_w, "carry_matrix": fx_carry},
            "metals":{"signals": mt_sig, "weights": mt_w, "zscores": mt_z},
            "sofr":  {"signals": sf_sig, "weights": sf_w, "slopes": sf_slopes},
        }


carry = CarryStrategy(params, TARGET_VOL, VOL_LAMBDA, LEV_CAP, REBAL_FREQ)
print("CarryStrategy initialised with 3 sub-modules.")

## Generate Signals (IS Period)

Run all three carry sub-modules and inspect signal diagnostics.

In [ ]:
results = carry.run_all(fx_data, metals_prices, sofr_prices, vix)

# ── Apply trend filter ───────────────────────────────────────────
# FX: trend column = pair name
fx_trend_map = {col: col for col in results["fx"]["weights"].columns
                if col in trend_filter_daily.columns}
# Metals: map spread to underlying front contract trend
metals_trend_map = {}
for col in results["metals"]["weights"].columns:
    if col.startswith("GC"):
        metals_trend_map[col] = "gc_fut_front"
    elif col.startswith("SI"):
        metals_trend_map[col] = "si_fut_front"
# SOFR: map to sofr_fut_front
sofr_trend_map = {col: "sofr_fut_front" for col in results["sofr"]["weights"].columns
                  if "sofr_fut_front" in trend_filter_daily.columns}

# Pure carry weights (before trend filter)
fx_w_pure   = results["fx"]["weights"].copy()
mt_w_pure   = results["metals"]["weights"].copy()
sf_w_pure   = results["sofr"]["weights"].copy()

# Carry-with-trend weights
fx_w_trend  = apply_trend_filter(fx_w_pure, trend_filter_daily, fx_trend_map, TREND_FILTER_MODE)
mt_w_trend  = apply_trend_filter(mt_w_pure, trend_filter_daily, metals_trend_map, TREND_FILTER_MODE)
sf_w_trend  = apply_trend_filter(sf_w_pure, trend_filter_daily, sofr_trend_map, TREND_FILTER_MODE)

# ── Signal diagnostics ───────────────────────────────────────────
for name, sigs in [("FX Carry", results["fx"]["signals"]),
                   ("Metals Spreads", results["metals"]["signals"]),
                   ("SOFR Curve", results["sofr"]["signals"])]:
    print(f"\n{'='*60}")
    print(f"  {name} — SIGNAL DIAGNOSTICS")
    print(f"{'='*60}")
    if sigs.empty:
        print("  No signals")
        continue
    for col in sigs.columns:
        s = sigs[col]
        n = len(s)
        pct_pos  = (s > 0).sum() / n * 100
        pct_neg  = (s < 0).sum() / n * 100
        pct_flat = (s == 0).sum() / n * 100
        changes  = (s.diff().abs() > 1e-10).sum()
        avg_hold = n / max(changes, 1)
        print(f"  {col:15s}  +={pct_pos:4.1f}%  -={pct_neg:4.1f}%  "
              f"flat={pct_flat:4.1f}%  changes={changes}  hold={avg_hold:.0f}d")

# Weight change frequency (should be weekly = ~52/year)
print(f"\n{'='*60}")
print(f"  WEIGHT CHANGE FREQUENCY (expect ~52/year)")
print(f"{'='*60}")
for name, w in [("FX", fx_w_trend), ("Metals", mt_w_trend), ("SOFR", sf_w_trend)]:
    if w.empty:
        continue
    for col in w.columns:
        changes = (w[col].diff().abs() > 1e-10).sum()
        n_years = len(w) / 252
        per_year = changes / max(n_years, 0.01)
        print(f"  {name:8s} {col:15s}  {changes} total  ({per_year:.0f}/year)")

## Backtest Engines (carry-specific)

Two backtest variants:
- **`backtest_weights`**: For FX carry (weights on spot returns)
- **`backtest_spread`**: For metals spreads and SOFR slopes (weights on spread changes)

Both enforce piecewise-constant weights (weekly frozen) and apply TC on changes.

In [ ]:
def backtest_weights(
    prices: pd.DataFrame,
    weights: pd.DataFrame,
    tc_bp: float = 2.0,
    capital: float = 1_000_000.0,
) -> pd.DataFrame:
    """Backtest using weights on price returns."""
    ret = prices.pct_change().fillna(0.0)
    # Align
    common_cols = [c for c in weights.columns if c in ret.columns]
    w = weights[common_cols].reindex(ret.index).ffill().fillna(0.0)
    r = ret[common_cols]

    port_ret_gross = (w.shift(1) * r).sum(axis=1)
    turnover = (w - w.shift(1)).abs().sum(axis=1).fillna(0.0)
    tc = turnover * (tc_bp / 10_000)
    port_ret_net = port_ret_gross - tc
    equity = capital * (1 + port_ret_net).cumprod()

    return pd.DataFrame({
        "ret_gross": port_ret_gross, "ret_net": port_ret_net,
        "equity": equity, "turnover": turnover,
    })


def backtest_spread(
    spreads: pd.DataFrame,
    weights: pd.DataFrame,
    tc_bp: float = 2.0,
    capital: float = 1_000_000.0,
) -> pd.DataFrame:
    """Backtest spread trades (PnL from spread changes)."""
    common_cols = [c for c in weights.columns if c in spreads.columns]
    if not common_cols:
        return pd.DataFrame()
    w = weights[common_cols].reindex(spreads.index).ffill().fillna(0.0)
    spread_ret = spreads[common_cols].pct_change().fillna(0.0)

    port_ret_gross = (w.shift(1) * spread_ret).sum(axis=1)
    turnover = (w - w.shift(1)).abs().sum(axis=1).fillna(0.0)
    tc = turnover * (tc_bp / 10_000)
    port_ret_net = port_ret_gross - tc
    equity = capital * (1 + port_ret_net).cumprod()

    return pd.DataFrame({
        "ret_gross": port_ret_gross, "ret_net": port_ret_net,
        "equity": equity, "turnover": turnover,
    })


def compute_metrics(bt: pd.DataFrame, periods: int = 252) -> dict:
    """Compute standard performance metrics."""
    r = bt["ret_net"].dropna()
    eq = bt["equity"].dropna()
    n_years = len(r) / periods
    total = (1 + r).prod()
    ann_ret = total ** (1 / max(n_years, 0.01)) - 1
    ann_vol = r.std() * np.sqrt(periods)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0.0
    rm = eq.cummax(); dd = (eq - rm) / rm
    max_dd = float(-dd.min()) if len(dd) > 0 else 0.0
    calmar = ann_ret / max_dd if max_dd > 0 else 0.0
    hit    = float((r > 0).sum() / len(r)) if len(r) > 0 else 0.0
    ann_to = bt["turnover"].sum() / max(n_years, 0.01)
    return {"Ann. Return": ann_ret, "Ann. Vol": ann_vol, "Sharpe": sharpe,
            "Max DD": max_dd, "Calmar": calmar, "Hit Rate": hit,
            "Ann. Turnover": ann_to}


print("Backtest engines ready.")

## Results — Pure Carry vs Carry-with-Trend

Compare each sub-module both ways:
1. **Pure carry:** Signals without trend filter
2. **Carry-with-trend:** TSMOM sign used as soft filter (0.5× when misaligned)

In [ ]:
# Build spread series for metals and SOFR
metals_spreads = pd.DataFrame(index=metals_prices.index)
spread_pairs = {
    "GC_1v2": ("gc_fut_front", "gc_fut_second"),
    "GC_2v3": ("gc_fut_second", "gc_fut_third"),
    "SI_1v2": ("si_fut_front", "si_fut_second"),
    "SI_2v3": ("si_fut_second", "si_fut_third"),
}
for label, (f, b) in spread_pairs.items():
    if f in metals_prices.columns and b in metals_prices.columns:
        metals_spreads[label] = metals_prices[f] - metals_prices[b]

sofr_slopes = pd.DataFrame(index=sofr_prices.index)
sofr_contracts = [c for c in sofr_cfg["instruments"] if c in sofr_prices.columns]
for i in range(len(sofr_contracts) - 1):
    sofr_slopes[f"SOFR_{i+1}v{i+2}"] = sofr_prices[sofr_contracts[i+1]] - sofr_prices[sofr_contracts[i]]
if len(sofr_contracts) >= 3:
    sofr_slopes["SOFR_fly"] = sofr_prices[sofr_contracts[0]] - 2*sofr_prices[sofr_contracts[1]] + sofr_prices[sofr_contracts[2]]

# ── Run backtests: Pure vs Trend ─────────────────────────────────
bt_results = {}
metrics_rows = []

for label, w_pure, w_trend, prices_or_spreads, bt_fn in [
    ("FX Carry", fx_w_pure, fx_w_trend, fx_spot_df, backtest_weights),
    ("Metals Spreads", mt_w_pure, mt_w_trend, metals_spreads, backtest_spread),
    ("SOFR Curve", sf_w_pure, sf_w_trend, sofr_slopes, backtest_spread),
]:
    for suffix, w in [("pure", w_pure), ("trend", w_trend)]:
        if w.empty:
            continue
        bt = bt_fn(prices_or_spreads, w, tc_bp=TC_BP)
        if bt.empty:
            continue
        key = f"{label} ({suffix})"
        bt_results[key] = bt
        m = compute_metrics(bt)
        m["Strategy"] = key
        metrics_rows.append(m)

comp_df = pd.DataFrame(metrics_rows).set_index("Strategy")

# Format
fmt_comp = comp_df.copy()
for c in ["Ann. Return", "Ann. Vol", "Max DD", "Hit Rate"]:
    fmt_comp[c] = fmt_comp[c].map("{:.1%}".format)
fmt_comp["Sharpe"]  = fmt_comp["Sharpe"].map("{:.2f}".format)
fmt_comp["Calmar"]  = fmt_comp["Calmar"].map("{:.2f}".format)
fmt_comp["Ann. Turnover"] = fmt_comp["Ann. Turnover"].map("{:.1f}x".format)

print("=" * 75)
print("  PURE CARRY vs CARRY-WITH-TREND")
print("=" * 75)
display(fmt_comp)

# ── Check: trend filter should reduce DD or improve Sharpe ───────
print("\nTrend filter impact:")
for mod in ["FX Carry", "Metals Spreads", "SOFR Curve"]:
    pure_key = f"{mod} (pure)"
    trend_key = f"{mod} (trend)"
    if pure_key in comp_df.index and trend_key in comp_df.index:
        d_sharpe = comp_df.loc[trend_key, "Sharpe"] - comp_df.loc[pure_key, "Sharpe"]
        d_dd = comp_df.loc[trend_key, "Max DD"] - comp_df.loc[pure_key, "Max DD"]
        print(f"  {mod:20s}  Sharpe delta: {d_sharpe:+.2f}  MaxDD delta: {d_dd:+.1%}")

# ── TSMOM correlation ────────────────────────────────────────────
import glob
tsmom_eq_files = sorted(glob.glob(str(Path("../outputs") / "tsmom_portfolio_equity_*.csv")))
if tsmom_eq_files:
    tsmom_port = pd.read_csv(tsmom_eq_files[-1], index_col=0, parse_dates=True)
    if "portfolio_ret_net" in tsmom_port.columns:
        tsmom_ret = tsmom_port["portfolio_ret_net"]
        print("\nCorrelation with TSMOM sleeve:")
        for key, bt in bt_results.items():
            if "trend" in key:
                corr = bt["ret_net"].corr(tsmom_ret.reindex(bt.index))
                print(f"  {key:30s}  rho = {corr:.3f}")
else:
    print("\nTSMOM equity file not found — skipping correlation check.")

## Combined Carry Sleeve

Combine FX + Metals + SOFR carry sub-modules (carry-with-trend variants)
using equal risk weighting, then apply portfolio-level vol targeting overlay
to 10% annual (sampled weekly, frozen within week).

In [ ]:
# ── Combine sub-module returns ────────────────────────────────────
sub_returns = {}
for name, w, prices_or_spreads, bt_fn in [
    ("fx", fx_w_trend, fx_spot_df, backtest_weights),
    ("metals", mt_w_trend, metals_spreads, backtest_spread),
    ("sofr", sf_w_trend, sofr_slopes, backtest_spread),
]:
    if w.empty:
        continue
    bt = bt_fn(prices_or_spreads, w, tc_bp=TC_BP)
    if not bt.empty:
        sub_returns[name] = bt["ret_net"]

if not sub_returns:
    print("No sub-module returns available!")
else:
    # Equal-weight across sub-modules
    sub_ret_df = pd.DataFrame(sub_returns).fillna(0.0)
    n_active = (sub_ret_df != 0).sum(axis=1).clip(lower=1)
    carry_ret_raw = sub_ret_df.sum(axis=1) / sub_ret_df.shape[1]

    # ── Portfolio vol targeting overlay (weekly frozen) ───────────
    carry_vol_rolling = carry_ret_raw.rolling(60, min_periods=20).std() * np.sqrt(252)
    scale = (TARGET_VOL / carry_vol_rolling.replace(0, np.nan)).clip(0.5, 2.0).fillna(1.0)

    # Freeze weekly
    scale_weekly = scale.resample(REBAL_FREQ).last()
    scale_daily = scale_weekly.reindex(carry_ret_raw.index).ffill().fillna(1.0)

    carry_ret_scaled = carry_ret_raw * scale_daily
    carry_equity = 1_000_000 * (1 + carry_ret_scaled).cumprod()

    # Combined turnover
    carry_turnover = pd.Series(0.0, index=carry_ret_raw.index)
    for name in sub_returns:
        key_trend = f"{'FX Carry' if name == 'fx' else 'Metals Spreads' if name == 'metals' else 'SOFR Curve'} (trend)"
        if key_trend in bt_results:
            carry_turnover += bt_results[key_trend]["turnover"].reindex(carry_turnover.index).fillna(0)

    # Metrics
    carry_bt = pd.DataFrame({
        "ret_net": carry_ret_scaled,
        "equity": carry_equity,
        "turnover": carry_turnover,
    })
    carry_sleeve_metrics = compute_metrics(carry_bt)
    carry_sleeve_metrics["Strategy"] = "COMBINED CARRY SLEEVE"

    # Realised vol check
    carry_vol_post = carry_ret_scaled.rolling(60, min_periods=20).std() * np.sqrt(252)

    print("=" * 65)
    print("  COMBINED CARRY SLEEVE METRICS (IS 2015-2022)")
    print("=" * 65)
    for k, v in carry_sleeve_metrics.items():
        if k == "Strategy":
            continue
        if isinstance(v, float):
            if "Return" in k or "Vol" in k or "DD" in k or "Rate" in k:
                print(f"  {k:20s}: {v:.1%}")
            elif "Turnover" in k:
                print(f"  {k:20s}: {v:.1f}x")
            else:
                print(f"  {k:20s}: {v:.2f}")

    print(f"\n  Realised vol (post-overlay):")
    print(f"    Mean   : {carry_vol_post.dropna().mean():.1%}  (target: {TARGET_VOL:.0%})")
    print(f"    Median : {carry_vol_post.dropna().median():.1%}")
    print(f"    Range  : [{carry_vol_post.dropna().min():.1%}, {carry_vol_post.dropna().max():.1%}]")

## Visualisations

1. Equity curves by sub-module + combined carry sleeve
2. FX carry ranking heatmap
3. Metals spread z-score heatmap
4. SOFR slope + signal panel
5. Turnover (weekly spikes)

In [ ]:
# --- 1. Equity Curves ---
fig_eq = go.Figure()
colors = {"FX Carry (trend)": "#3498db", "Metals Spreads (trend)": "#f39c12",
          "SOFR Curve (trend)": "#2ecc71"}
for key, bt in bt_results.items():
    if "trend" not in key:
        continue
    eq_norm = bt["equity"] / bt["equity"].iloc[0]
    fig_eq.add_trace(go.Scatter(
        x=eq_norm.index, y=eq_norm, name=key,
        line=dict(color=colors.get(key, "#95a5a6"), width=1.5),
    ))
if "carry_equity" in dir():
    eq_norm = carry_equity / carry_equity.iloc[0]
    fig_eq.add_trace(go.Scatter(
        x=eq_norm.index, y=eq_norm, name="Combined Sleeve",
        line=dict(color="#2c3e50", width=2.5),
    ))
fig_eq.update_layout(
    title="Carry Strategy — Equity Curves (IS: 2015-2022, carry-with-trend)",
    template="plotly_white", height=500, hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
    yaxis_title="Growth of $1", yaxis_tickformat="$.2f",
)
fig_eq.show()

# --- 2. FX Carry Ranking Heatmap ---
carry_matrix = results["fx"]["carry_matrix"]
if not carry_matrix.empty:
    carry_monthly = (carry_matrix * 10_000).resample("ME").last().dropna()
    fig_fx = px.imshow(
        carry_monthly.T.round(0),
        title="FX Implied Carry (bp annualised) — Monthly",
        labels={"x": "", "y": "Pair", "color": "Carry (bp)"},
        color_continuous_scale="RdYlGn", aspect="auto",
    )
    fig_fx.update_layout(template="plotly_white", height=300,
                         xaxis=dict(dtick="M6", tickformat="%Y-%m"))
    fig_fx.show()

# --- 3. Metals Spread Z-Score Heatmap ---
metals_z = results["metals"]["zscores"]
if not metals_z.empty:
    zs_monthly = metals_z.resample("ME").last().dropna()
    fig_mz = px.imshow(
        zs_monthly.T.round(2),
        title="Metals Calendar Spread Z-Scores — Monthly",
        labels={"x": "", "y": "Spread", "color": "Z-Score"},
        color_continuous_scale="RdBu_r", zmin=-3, zmax=3, aspect="auto",
    )
    fig_mz.add_annotation(
        text=f"Entry: +/-{met_cfg['entry_zscore']}s  |  Exit: +/-{met_cfg['exit_zscore']}s",
        xref="paper", yref="paper", x=0.01, y=1.08,
        showarrow=False, font=dict(size=11, color="gray"),
    )
    fig_mz.update_layout(template="plotly_white", height=300,
                         xaxis=dict(dtick="M6", tickformat="%Y-%m"))
    fig_mz.show()

# --- 4. SOFR Slope + Signal ---
sofr_sigs = results["sofr"]["signals"]
sofr_sl   = results["sofr"]["slopes"]  # note: from strategy, not local
if not sofr_sl.empty:
    fig_sofr = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.6, 0.4],
        subplot_titles=["SOFR Slopes (price space)", "Signals"],
        vertical_spacing=0.10,
    )
    for col in sofr_sl.columns:
        fig_sofr.add_trace(go.Scatter(
            x=sofr_sl.index, y=sofr_sl[col], name=col, line=dict(width=1.5),
        ), row=1, col=1)
    for col in sofr_sigs.columns:
        fig_sofr.add_trace(go.Scatter(
            x=sofr_sigs.index, y=sofr_sigs[col], name=f"{col} sig",
            line=dict(width=1, dash="dot"),
        ), row=2, col=1)
    fig_sofr.update_layout(
        title="SOFR Curve — Slopes & Signals", template="plotly_white",
        height=550, hovermode="x unified",
        legend=dict(orientation="h", y=-0.15),
    )
    fig_sofr.update_yaxes(title_text="Slope", row=1, col=1)
    fig_sofr.update_yaxes(title_text="Signal", range=[-1.2, 1.2], row=2, col=1)
    fig_sofr.show()

# --- 5. Turnover ---
if bt_results:
    total_turn = pd.Series(0.0, dtype=float)
    for key, bt in bt_results.items():
        if "trend" in key:
            total_turn = total_turn.add(bt["turnover"], fill_value=0.0)
    fig_turn = go.Figure()
    fig_turn.add_trace(go.Bar(
        x=total_turn.index, y=total_turn,
        marker_color="#3498db", opacity=0.7, name="Turnover",
    ))
    fig_turn.update_layout(
        title="Carry Sleeve — Daily Turnover (expect weekly spikes)",
        template="plotly_white", height=300,
        yaxis_title="Turnover |Dw|", hovermode="x unified",
    )
    fig_turn.show()

## Export

Save signals, equity curves, and summary to `outputs/`.

In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

datestamp = datetime.now().strftime("%Y%m%d")

# FX carry signals
p1 = output_dir / f"carry_fx_signals_{datestamp}.csv"
results["fx"]["signals"].to_csv(p1)

# Metals spread signals
p2 = output_dir / f"carry_metals_spreads_signals_{datestamp}.csv"
results["metals"]["signals"].to_csv(p2)

# SOFR signals
p3 = output_dir / f"carry_sofr_signals_{datestamp}.csv"
results["sofr"]["signals"].to_csv(p3)

# Equity curves
eq_export = pd.DataFrame()
for key, bt in bt_results.items():
    if "trend" in key:
        eq_export[key] = bt["equity"]
if "carry_equity" in dir():
    eq_export["Combined Sleeve"] = carry_equity
p4 = output_dir / f"carry_equity_curves_{datestamp}.csv"
eq_export.to_csv(p4)

# Carry sleeve daily returns (for portfolio integration)
p5 = output_dir / f"carry_sleeve_daily_returns_{datestamp}.csv"
if "carry_ret_scaled" in dir():
    carry_ret_scaled.to_csv(p5, header=["carry_sleeve_ret_net"])

# Summary HTML
p6 = output_dir / f"carry_summary_{datestamp}.html"
html = (
    "<h2>Carry Strategy — IS Performance (2015-2022)</h2>\n"
    "<p>FX cross-sectional carry + Metals calendar spreads + SOFR curve<br>"
    f"Trend filter: {TREND_FILTER_MODE} | Vol target: {TARGET_VOL:.0%} | TC: {TC_BP:.0f}bp</p>\n"
    "<h3>Pure Carry vs Carry-with-Trend</h3>\n"
    + fmt_comp.to_html()
)
if "carry_sleeve_metrics" in dir():
    html += "<br><h3>Combined Carry Sleeve</h3>\n<pre>"
    for k, v in carry_sleeve_metrics.items():
        if k == "Strategy": continue
        if isinstance(v, float):
            html += f"  {k:20s}: {v:.4f}\n"
    html += "</pre>"
with open(p6, "w") as f:
    f.write(html)

print(f"Exported to {output_dir.resolve()}/")
for p in [p1, p2, p3, p4, p5, p6]:
    print(f"  {p.name}")

print(f"\nNotebook complete: {datetime.now():%Y-%m-%d %H:%M}")